# exp-000 baseline-median — score floor + metric verification

In [ ]:
import os
os.environ["PYTHONHASHSEED"] = "42"
import json
import random
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import cloudpickle
from sklearn.model_selection import KFold


In [ ]:
PROJECT_ROOT = "../../.."
EXP_DIR = "."
SEED = 42
N_SPLITS = 5


In [ ]:
random.seed(SEED)
np.random.seed(SEED)

sys.path.insert(0, str(Path(PROJECT_ROOT).resolve()))
from tools.config import load_config, resolve_metric

root = Path(PROJECT_ROOT).resolve()
exp_dir = Path(EXP_DIR).resolve()
cfg = load_config(root)
mae = resolve_metric(cfg["task"]["metric"], root)


## Load

In [ ]:
raw = root / cfg["paths"]["raw"]
train = pd.read_csv(raw / "train.csv")
test = pd.read_csv(raw / "test.csv")
print(train.shape, test.shape)


## CV: median vs. mean constant predictor

Confirms the metric rewards the conditional median (MAE-optimal) rather than the mean (RMSE-optimal), before any text-based model is trusted.

In [ ]:
y = train["listPrice"].values
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

median_scores, mean_scores = [], []
for tr_idx, va_idx in kf.split(y):
    fold_median = float(np.median(y[tr_idx]))
    fold_mean = float(np.mean(y[tr_idx]))
    median_scores.append(mae(y[va_idx], np.full(len(va_idx), fold_median)))
    mean_scores.append(mae(y[va_idx], np.full(len(va_idx), fold_mean)))

cv_median_mae = float(np.mean(median_scores))
cv_mean_mae = float(np.mean(mean_scores))
print(f"median baseline cv_mae={cv_median_mae:.2f}")
print(f"mean baseline   cv_mae={cv_mean_mae:.2f}")
assert cv_median_mae < cv_mean_mae, "metric does not prefer the median -- investigate"


## Save & Load Model

The champion constant is the full-train median. Bundled with cloudpickle so the class definition travels with the file.

In [ ]:
class ConstantMedianModel:
    def __init__(self, constant):
        self.constant = constant

    def predict(self, **frames):
        test_df = frames["test"]
        return pd.DataFrame({
            "id": test_df["id"],
            "listPrice": np.full(len(test_df), self.constant),
        })

full_train_median = float(np.median(y))
model = ConstantMedianModel(full_train_median)

with open(exp_dir / "model.pkl", "wb") as fh:
    cloudpickle.dump(model, fh)


In [ ]:
with open(exp_dir / "model.pkl", "rb") as fh:
    loaded = cloudpickle.load(fh)
assert loaded.constant == model.constant


## Prediction

In [ ]:
pred = model.predict(test=test)
pred.to_csv(exp_dir / "submission.csv", index=False,
            float_format="%.6f", lineterminator="\n")

pred_loaded = loaded.predict(test=test)
pred_loaded.to_csv(exp_dir / "submission_check.csv", index=False,
                    float_format="%.6f", lineterminator="\n")
assert (pred["listPrice"] == pred_loaded["listPrice"]).all()

metrics_out = {
    "cv_primary": cv_median_mae,
    "cv_mean_baseline_for_reference": cv_mean_mae,
    "full_train_median": full_train_median,
}
print(json.dumps(metrics_out, indent=2))
with open(exp_dir / "cv_metrics.json", "w") as fh:
    json.dump(metrics_out, fh, indent=2)
